In [5]:
# task: review and allocate all data combinations within SMILES -> PDB
# 30k ish SMILES
# for each SMILES: many PDBs
# for each of those PDBs -> many context coordinates + Y
# need an allocator to generate training + test datasets for this task


# projected unpacking time = 41 minutes
# TODO: organize into train and test + batch processor

import torch
import re
import ast
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm


class MakePointer:

    def __init__(self, smilespdb, pdb_radial, smiles_molfp,
                 train_pct, seed):
        smilespdb["PDB_hits"] = smilespdb["PDB_hits"].apply(ast.literal_eval)
        self.smilespdb = smilespdb
        self.radial_data = pdb_radial
        self.molfp_data = smiles_molfp

        if not 0 < train_pct < 1:
            raise ValueError(f"Train % split must be between 0.00 and 1.00: value submitted {train_pct}")
        self.train_pct = train_pct
        self.seed = seed

        self.train_rows = None
        self.test_rows = None


# notes on data columns
# print(SMILE_2_PDBHITS.columns)  # 'SMILES', 'PDB_hits'
# print(MOLECULAR_FINGERPRINTS.columns)  # 'smiles_str', 'map4_fp'
# print(RADIAL_SEQUENCES.columns)  # 'PDB_ID', 'radial_sequence'

    def make_pointerfile(self, out_folder):
        rng = np.random.default_rng(self.seed)

        all_smiles = self.smilespdb["SMILES"].tolist()
        rng.shuffle(all_smiles)

        n_train = int(len(all_smiles) * self.train_pct)
        train_set = set(all_smiles[:n_train])
        test_set = set(all_smiles[n_train:])  # unseen smiles -> ZeroBind methodology

        train_rows, test_rows = [], []

        for _, row in tqdm(self.smilespdb.iterrows(), total=len(self.smilespdb), desc="Building Pointer Files"):
            smiles = row["SMILES"]
            pdb_hits = row["PDB_hits"]
            bucket = train_rows if smiles in train_set else test_rows

            for pdb_id in pdb_hits:
                seq_data = self.radial_data.loc[self.radial_data["PDB_ID"]==pdb_id]
                if not len(seq_data):
                    continue
                seq_len = len(seq_data["radial_sequence"].values[0])
                n_windows = seq_len

                for window_idx in range(n_windows):
                    bucket.append({
                        "SMILES": smiles,
                        "PDB_HIT": pdb_id,
                        "WINDOW_IDX": window_idx,
                    })


        self.train_rows = pd.DataFrame(train_rows)
        self.test_rows = pd.DataFrame(test_rows)

        out_path = Path(str(out_folder))
        self.train_rows.to_parquet(out_path / "training_pointers.parquet", index=False)
        self.test_rows.to_parquet(out_path / "test_pointers.parquet", index=False)

        return True

# TODO: function to sample some molecular fingerprints and calculate tanimoto similarity (1000 at a time?)


In [9]:
PROJECT_ROOT = Path.cwd().parent
DATAPATH = PROJECT_ROOT / "database"

In [ ]:
smiles_pdb = pd.read_csv(DATAPATH / "SMILE_2_PDBhits.csv")
pdb_radial = pd.read_pickle(DATAPATH / "radial_seq_df.pkl")
smiles_molfp = pd.read_pickle(DATAPATH / "molfp_df.pkl")

In [ ]:
handler = MakePointer(smilespdb=smiles_pdb, pdb_radial=pdb_radial,
                      smiles_molfp=smiles_molfp,
                      train_pct=0.8, seed=42)

In [4]:
print(handler.radial_data["radial_sequence"].iloc[0])

[[['I'], [61, 56, 32]], [['K'], [68, 53, 41]], [['M'], [54, 44, 32]], [['A'], [49, 34, 59]], [['M'], [59, 41, 36]], [['F'], [63, 41, 40]], [['Y'], [68, 49, 44]], [['P'], [67, 57, 52]], [['N'], [63, 48, 37]], [['Y'], [56, 61, 37]], [['A'], [64, 42, 57]], [['D'], [50, 41, 36]], [['Y'], [64, 56, 42]], [['G'], [46, 65, 46]], [['A'], [61, 39, 49]], [['Y'], [53, 36, 56]], [['A'], [51, 34, 50]], [['Y'], [45, 38, 43]], [['A'], [39, 60, 46]], [['K'], [42, 42, 40]], [['V'], [46, 60, 40]], [['L'], [51, 64, 44]], [['H'], [63, 52, 57]], [['M'], [59, 39, 55]], [['P'], [45, 47, 37]], [['N'], [63, 55, 48]], [['V'], [64, 46, 52]], [['C'], [62, 48, 43]], [['E'], [59, 48, 59]], [['W'], [46, 38, 51]], [['D'], [43, 55, 41]], [['L'], [44, 60, 48]], [['F'], [50, 40, 44]], [['V'], [47, 51, 40]], [['V'], [58, 46, 54]], [['P'], [48, 43, 49]], [['VOID'], [0, 0, 0]]]


In [5]:
out_folder = "notebook_database"
(handler.make_pointerfile(out_folder=out_folder))

Building Pointer Files: 100%|██████████| 5036/5036 [01:42<00:00, 49.37it/s] 


True

In [6]:
print(handler.train_rows.head(3))
print(handler.train_rows["WINDOW_IDX"].unique())

print(handler.test_rows.head(3))
print(handler.test_rows["WINDOW_IDX"].unique())

                                           SMILES PDB_HIT  WINDOW_IDX
0  B(CNS(=O)(=O)c1ccc(cc1C(F)(F)F)c2[nH]nnn2)(O)O    9DHL           0
1  B(CNS(=O)(=O)c1ccc(cc1C(F)(F)F)c2[nH]nnn2)(O)O    9DHL           1
2  B(CNS(=O)(=O)c1ccc(cc1C(F)(F)F)c2[nH]nnn2)(O)O    9DHL           2
[  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17
  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35
  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53
  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71
  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89
  90  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107]
                                      SMILES PDB_HIT  WINDOW_IDX
0  Brc1ccc(cn1)N1CC2CC(C1)N2 |TLB:4:7:13:10|    8V8A           0
1  Brc1ccc(cn1)N1CC2CC(C1)N2 |TLB:4:7:13:10|    8V8A           1
2  Brc1ccc(cn1)N1CC2CC(C1)N2 |TLB:4:7:13:10|    8V8A           2
[  0   1   2   3   4 

In [ ]:
pdb_radial = pd.read_pickle(DATAPATH / "radial_seq_df.pkl")
smiles_molfp = pd.read_pickle(DATAPATH / "molfp_df.pkl")
train_pointer = pd.read_parquet(DATAPATH / "training_pointers.parquet")
test_pointer = pd.read_parquet(DATAPATH / "test_pointers.parquet")

In [ ]:
# inspect parquet files
print(train_pointer.columns)
print(test_pointer.columns)

In [1]:
# turns out google colab really doesn't like parquet files
import pyarrow.parquet as pq
import pyarrow as pa

In [12]:
training_pointer = DATAPATH / "test_pointers.parquet"

table = pq.read_table(training_pointer)

table = table.set_column(
    table.schema.get_field_index("SMILES"),
    "SMILES",
    table.column("SMILES").dictionary_encode()
)
table = table.set_column(
    table.schema.get_field_index("PDB_HIT"),
    "PDB_HIT",
    table.column("PDB_HIT").dictionary_encode()
)

pq.write_table(table, DATAPATH / "test_pointers_dict.parquet", compression="zstd")